# E001 — Signal survey on the training split

**Hypotheses:** H2, H5, H6, H7, H8 ([docs/hypotheses.md](../../docs/hypotheses.md))

**Question:** Which raw signals move CTR before any model, and which of them would leak?

**Method:** Group-by CTR tables on the training split only (validation and test unseen), plus the genre/tier x campaign interaction residuals against their binomial noise.

In [1]:
from IPython.display import Markdown

from charade.analysis.experiment import setup

ctx = setup()
f"mode: {'smoke (fixture, tiny model)' if ctx.smoke else 'full data'}"

'mode: full data'

## Run
Every table is computed on the training split (10-21..10-27) except the daily CTR series.

In [2]:
from charade.analysis import eda

eda.run(ctx.data_dir, ctx.reports / "eda")
Markdown((ctx.reports / "eda" / "eda.md").read_text())

# EDA (generated by `uv run poe eda`; training split unless stated)

## Daily CTR (all splits)

| day | split | impressions | ctr |
|---|---|---|---|
| 2014-10-21 | train | 111,065 | 0.1792 |
| 2014-10-22 | train | 143,836 | 0.1666 |
| 2014-10-23 | train | 104,452 | 0.1916 |
| 2014-10-24 | train | 89,378 | 0.1843 |
| 2014-10-25 | train | 90,218 | 0.1925 |
| 2014-10-26 | train | 103,948 | 0.1950 |
| 2014-10-27 | train | 86,788 | 0.1955 |
| 2014-10-28 | val | 142,909 | 0.1654 |
| 2014-10-29 | test | 104,450 | 0.1727 |
| 2014-10-30 | test | 22,956 | 0.1655 |

## Genre

| genre | impressions | ctr |
|---|---|---|
| mentor | 92,920 | 0.1467 |
| romance | 76,370 | 0.2315 |
| slice | 76,104 | 0.1776 |
| comedic | 74,272 | 0.1820 |
| anime | 71,391 | 0.1817 |
| historical | 69,549 | 0.1797 |
| mystery | 68,399 | 0.1800 |
| horror | 67,770 | 0.2248 |
| sci | 67,035 | 0.1783 |
| fantasy | 65,875 | 0.1769 |

## Safety tier

| safety_tier | impressions | ctr |
|---|---|---|
| sfw | 429,741 | 0.1806 |
| suggestive | 190,589 | 0.1830 |
| mature | 109,355 | 0.2054 |

## Creator type

| creator_type | impressions | ctr |
|---|---|---|
| community | 647,755 | 0.1850 |
| official | 81,930 | 0.1846 |

## Surface

| surface | impressions | ctr |
|---|---|---|
| site | 479,078 | 0.2112 |
| app | 250,607 | 0.1348 |

## Banner position

| banner_pos | impressions | ctr |
|---|---|---|
| 0 | 531,339 | 0.1795 |
| 1 | 197,073 | 0.1990 |
| 7 | 687 | 0.4221 |
| 2 | 251 | 0.1713 |
| 5 | 166 | 0.1145 |
| 4 | 153 | 0.1569 |
| 3 | 16 | 0.1875 |

## Conversation turn

| turn_bucket | impressions | ctr |
|---|---|---|
| <=1 | 160,532 | 0.1836 |
| <=10 | 141,920 | 0.1856 |
| <=2 | 102,060 | 0.1846 |
| <=20 | 100,295 | 0.1838 |
| <=3 | 74,815 | 0.1867 |
| <=5 | 105,945 | 0.1861 |
| >20 | 44,118 | 0.1859 |

## Placeholder device_id share

| share |
|---|
| 0.8253 |

## User seen before (strictly earlier hours)

| prior_imps | impressions | ctr |
|---|---|---|
| 0 | 595,455 | 0.1908 |
| 1 | 50,165 | 0.1528 |
| 2 | 17,892 | 0.1567 |
| 3 | 9,320 | 0.1548 |
| 4 | 6,131 | 0.1501 |
| 5 | 50,722 | 0.1682 |

## Same-hour user impressions (LEAK: unknown at serve time)

| same_hour_user_imps | impressions | ctr |
|---|---|---|
| 1 | 633,327 | 0.1929 |
| 2 | 56,074 | 0.1429 |
| 3 | 14,100 | 0.1172 |
| 4 | 6,832 | 0.1071 |
| 5 | 4,405 | 0.0981 |
| 6 | 14,947 | 0.1294 |

## Prior exposures to the same campaign (row order within hour)

| prior_same_campaign | impressions | ctr |
|---|---|---|
| 0 | 596,687 | 0.1948 |
| 1 | 59,831 | 0.1413 |
| 2 | 18,653 | 0.1334 |
| 3 | 9,464 | 0.1301 |
| 4 | 6,109 | 0.1331 |
| 5 | 38,941 | 0.1481 |

## Character x campaign interaction

| factor | cells | residual_sd | mean_binomial_se |
|---|---|---|---|
| genre | 300 | 0.0274 | 0.0111 |
| safety_tier | 90 | 0.0156 | 0.0069 |


## Reading
- Genre spans mentor ~14.7 % to romance ~23.2 %; mature tier sits ~2.5 pp above sfw.
- Conversation-turn buckets all sit at 18.4-18.7 %: no visible signal (H7).
- Users seen in earlier hours click less (15-17 % vs 19.1 %); repeat campaign exposure drops CTR from
  ~19.5 % to 13-15 % (H6).
- Same-hour user counts separate CTR sharply, but the hour's total is only known after the hour (H5).
- Genre x campaign residual sd 2.7 pp vs 1.1 pp binomial noise: interactions are real (feeds H1).

## Conclusion
Genre, safety tier, prior user activity and campaign repetition carry signal; conversation turn and character age do not; same-hour user counts are a leak.

**Decision:** Keep genre, tier and causal user counters; treat turn and age as ablation candidates; exclude same-hour counts.